In [1]:
!conda list

# packages in environment at /Users/alexandru/miniconda3/envs/thesisenv:
#
# Name                     Version          Build               Channel
aom                        3.12.1           ha237518_0
appnope                    0.1.4            pyhd8ed1ab_1        conda-forge
artist                     0.18.2           pypi_0              pypi
asttokens                  3.0.1            pyhd8ed1ab_0        conda-forge
blas                       1.0              openblas
brotlicffi                 1.1.0.0          py310h50f4ffc_0
bzip2                      1.0.8            hd037594_8          conda-forge
ca-certificates            2025.11.12       hbd8a1cb_0          conda-forge
cairo                      1.18.4           h191e429_0
certifi                    2025.11.12       py310hca03da5_0
cffi                       2.0.0            py310h73c2a22_1
charset-normalizer         3.4.4            py310hca03da5_0
comm                       0.2.3            pyhe01879c_0        conda-forge
d

In [1]:
import artist



In [3]:
import pathlib

import torch

import artist


from artist.data_parser import paint_scenario_parser
from artist.scenario.configuration_classes import (
    LightSourceConfig,
    LightSourceListConfig,
)
from artist.scenario.h5_scenario_generator import H5ScenarioGenerator
from artist.util import config_dictionary, set_logger_config
from artist.util.environment_setup import get_device

# Set up logger
set_logger_config()

torch.manual_seed(7)
torch.cuda.manual_seed(7)

# Set the device.
device = get_device()

# Specify the path to your scenario file.
scenario_path = pathlib.Path("tutorials/data/scenarios/single_heliostat_scenario.h5")

# Specify the path to your tower-measurements.json file.
tower_file = pathlib.Path(
    "tutorials/data/paint/tower-measurements.json"
)

# Specify the following data for each heliostat that you want to include in the scenario:
# A tuple of: (heliostat-name, heliostat-properties.json, deflectometry.h5)
heliostat_files_list = [
    (
        "name",
        pathlib.Path(
            "tutorials/data/paint/AA28/heliostat-properties.json"
        ),
        pathlib.Path("tutorials/data/paint/AA28/deflectometry.h5"),
    ),
    # (
    #     "name2",
    #     pathlib.Path(
    #         "please/insert/the/path/to/the/paint/data/here/heliostat-properties.json"
    #     ),
    #     pathlib.Path("please/insert/the/path/to/the/paint/data/here/deflectometry.h5"),
    # ),
    # # ... Include as many as you want, but at least one!
]

# This checks to make sure the path you defined is valid and a scenario HDF5 can be saved there.
if not pathlib.Path(scenario_path).parent.is_dir():
    raise FileNotFoundError(
        f"The folder ``{pathlib.Path(scenario_path).parent}`` selected to save the scenario does not exist. "
        "Please create the folder or adjust the file path before running again!"
    )

# Include the power plant configuration.
power_plant_config, target_area_list_config = (
    paint_scenario_parser.extract_paint_tower_measurements(
        tower_measurements_path=tower_file, device=device
    )
)

# Include the light source configuration.
light_source1_config = LightSourceConfig(
    light_source_key="sun_1",
    light_source_type=config_dictionary.sun_key,
    number_of_rays=10,
    distribution_type=config_dictionary.light_source_distribution_is_normal,
    mean=0.0,
    covariance=4.3681e-06,
)

# Create a list of light source configs - in this case only one.
light_source_list = [light_source1_config]

# Include the configuration for the list of light sources.
light_source_list_config = LightSourceListConfig(light_source_list=light_source_list)

target_area = [
    target_area
    for target_area in target_area_list_config.target_area_list
    if target_area.target_area_key == config_dictionary.target_area_receiver
]

number_of_nurbs_control_points = torch.tensor([20, 20], device=device)
nurbs_fit_method = config_dictionary.fit_nurbs_from_normals
nurbs_deflectometry_step_size = 100
nurbs_fit_tolerance = 1e-10
nurbs_fit_max_epoch = 400

# Please leave the optimizable parameters empty, they will automatically be added for the surface fit.
nurbs_fit_optimizer = torch.optim.Adam([torch.empty(1, requires_grad=True)], lr=1e-3)
nurbs_fit_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    nurbs_fit_optimizer,
    mode="min",
    factor=0.2,
    patience=50,
    threshold=1e-7,
    threshold_mode="abs",
)

heliostat_list_config, prototype_config = (
    paint_scenario_parser.extract_paint_heliostats_fitted_surface(
        paths=heliostat_files_list,
        power_plant_position=power_plant_config.power_plant_position,
        number_of_nurbs_control_points=number_of_nurbs_control_points,
        deflectometry_step_size=nurbs_deflectometry_step_size,
        nurbs_fit_method=nurbs_fit_method,
        nurbs_fit_tolerance=nurbs_fit_tolerance,
        nurbs_fit_max_epoch=nurbs_fit_max_epoch,
        nurbs_fit_optimizer=nurbs_fit_optimizer,
        nurbs_fit_scheduler=nurbs_fit_scheduler,
        device=device,
    )
)

if __name__ == "__main__":
    """Generate the scenario given the defined parameters."""
    scenario_generator = H5ScenarioGenerator(
        file_path=scenario_path,
        power_plant_config=power_plant_config,
        target_area_list_config=target_area_list_config,
        light_source_list_config=light_source_list_config,
        prototype_config=prototype_config,
        heliostat_list_config=heliostat_list_config,
    )
    scenario_generator.generate_scenario()


[2025-11-27 20:05:51,205][artist.util.environment_setup][INFO] - No device type provided. The device will default to GPU based on availability and OS, otherwise to CPU.
[2025-11-27 20:05:51,205][artist.util.environment_setup][INFO] - No device type provided. The device will default to GPU based on availability and OS, otherwise to CPU.
[2025-11-27 20:05:51,205][artist.util.environment_setup][WARNING] - Setting device to CPU. ARTIST only supports CPU for MacOS.
[2025-11-27 20:05:51,205][artist.util.environment_setup][WARNING] - Setting device to CPU. ARTIST only supports CPU for MacOS.
[2025-11-27 20:05:51,206][artist.data_parser.paint_scenario_parser][INFO] - Beginning extraction of tower data from PAINT file.
[2025-11-27 20:05:51,206][artist.data_parser.paint_scenario_parser][INFO] - Beginning extraction of tower data from PAINT file.
[2025-11-27 20:05:51,247][artist.data_parser.paint_scenario_parser][INFO] - Loading tower data complete.
[2025-11-27 20:05:51,247][artist.data_parser.pa